In [1]:
import os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
PRETRAIN_DATA = os.path.join(REPO_ROOT, "data", "3.pretraining", "1.dap")

import os
import torch

# 검사 및 수정할 체크포인트 경로 지정
checkpoint_dir = os.path.join(PRETRAIN_DATA, "checkpoint-156600")
bin_file_path = os.path.join(checkpoint_dir, "pytorch_model.bin")
safe_file_path = os.path.join(checkpoint_dir, "model.safetensors")

# 1. 파일 로드 및 포맷 확인
is_safetensors = False
if os.path.exists(safe_file_path):
    from safetensors.torch import load_file
    state_dict = load_file(safe_file_path)
    is_safetensors = True
    original_path = safe_file_path
    new_path = os.path.join(checkpoint_dir, "model_fixed.safetensors")
    print(f"[{original_path}] safetensors 파일을 로드했습니다.\n")
elif os.path.exists(bin_file_path):
    state_dict = torch.load(bin_file_path, map_location="cpu")
    original_path = bin_file_path
    new_path = os.path.join(checkpoint_dir, "pytorch_model_fixed.bin")
    print(f"[{original_path}] pytorch_model.bin 파일을 로드했습니다.\n")
else:
    raise FileNotFoundError("체크포인트 디렉토리에서 가중치 파일을 찾을 수 없습니다.")

# 2. 원본 키 값 검사 및 출력
print("--- 원본 파일 내 LayerNorm 관련 키 검사 ---")
ln_keys_original = [k for k in state_dict.keys() if "LayerNorm" in k]
for i, key in enumerate(ln_keys_original[:5]): # 예시로 상위 5개만 출력
    print(f"발견된 원본 키 [{i+1}]: {key}")
print(f"... (총 {len(ln_keys_original)}개의 LayerNorm 키 발견)\n")

# 3. 파라미터 이름 변환 
new_state_dict = {}
change_count = 0

for key, value in state_dict.items():
    new_key = key
    if "LayerNorm.gamma" in key:
        new_key = key.replace("LayerNorm.gamma", "LayerNorm.weight")
        change_count += 1
    elif "LayerNorm.beta" in key:
        new_key = key.replace("LayerNorm.beta", "LayerNorm.bias")
        change_count += 1
    
    new_state_dict[new_key] = value

# 4. 변경된 키 확인 및 새로운 파일로 저장
if change_count > 0:
    print("--- 변경될 파일 내 LayerNorm 관련 키 예시 ---")
    ln_keys_new = [k for k in new_state_dict.keys() if "LayerNorm" in k]
    for i, key in enumerate(ln_keys_new[:5]):
        print(f"변환된 새 키 [{i+1}]: {key}")
        
    print(f"\n총 {change_count}개의 파라미터 이름이 변경 대상입니다.")
    
    # 새로운 파일로 분리하여 저장 (원본 보존)
    if is_safetensors:
        from safetensors.torch import save_file
        save_file(new_state_dict, new_path)
    else:
        torch.save(new_state_dict, new_path)
    print(f"수정된 가중치를 새로운 파일로 저장 완료했습니다: {new_path}")
    print("\n[적용 방법] 기존 원본 파일의 이름을 'pytorch_model_old.bin' 등으로 변경하고, 새로 생성된 파일의 이름에서 '_fixed'를 제거하여 원본 파일명과 동일하게 맞춰주십시오.")
else:
    print("\n변경이 필요한 LayerNorm.gamma / beta 파라미터가 없습니다. 원본 파일이 이미 최신 명명 규칙을 따르고 있습니다.")

[./outputs/checkpoint-156600/model.safetensors] safetensors 파일을 로드했습니다.

--- 원본 파일 내 LayerNorm 관련 키 검사 ---
발견된 원본 키 [1]: bert.embeddings.LayerNorm.beta
발견된 원본 키 [2]: bert.embeddings.LayerNorm.gamma
발견된 원본 키 [3]: bert.encoder.layer.0.attention.output.LayerNorm.beta
발견된 원본 키 [4]: bert.encoder.layer.0.attention.output.LayerNorm.gamma
발견된 원본 키 [5]: bert.encoder.layer.0.output.LayerNorm.beta
... (총 52개의 LayerNorm 키 발견)

--- 변경될 파일 내 LayerNorm 관련 키 예시 ---
변환된 새 키 [1]: bert.embeddings.LayerNorm.bias
변환된 새 키 [2]: bert.embeddings.LayerNorm.weight
변환된 새 키 [3]: bert.encoder.layer.0.attention.output.LayerNorm.bias
변환된 새 키 [4]: bert.encoder.layer.0.attention.output.LayerNorm.weight
변환된 새 키 [5]: bert.encoder.layer.0.output.LayerNorm.bias

총 52개의 파라미터 이름이 변경 대상입니다.
수정된 가중치를 새로운 파일로 저장 완료했습니다: ./outputs/checkpoint-156600/model_fixed.safetensors

[적용 방법] 기존 원본 파일의 이름을 'pytorch_model_old.bin' 등으로 변경하고, 새로 생성된 파일의 이름에서 '_fixed'를 제거하여 원본 파일명과 동일하게 맞춰주십시오.
